https://tianchi.aliyun.com/notebook/144454

# 排序模型



通过召回的操作， 我们已经进行了问题规模的缩减， 对于每个用户， 选择出了N篇文章作为了候选集，并基于召回的候选集构建了与用户历史相关的特征，以及用户本身的属性特征，文章本省的属性特征，以及用户与文章之间的特征，下面就是使用机器学习模型来对构造好的特征进行学习，然后对测试集进行预测，得到测试集中的每个候选集用户点击的概率，返回点击概率最大的topk个文章，作为最终的结果

排序阶段选择了三个比较有代表性的排序模型，它们分别是:

- LGB的排序模型
- LGB的分类模型
- 深度学习的分类模型DIN

得到了最终的排序模型输出的结果之后，还选择了两种比较经典的模型集成的方法:

- 输出结果加权融合
- Staking（将模型的输出结果再使用一个简单模型进行预测

In [1]:
import numpy as np
import pandas as pd
import pickle
from tqdm import tqdm
import gc, os
import time
from datetime import datetime
import lightgbm as lgb
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

In [2]:
import os
import sys

# 导入项目公共模块
sys.path.insert(0, os.path.abspath(".."))

from metric.data import prepare_evaluation
from metric.evaluation import (
    evaluate_predictions,
    evaluate_csv,
    save_predictions
)

# 路径及配置

offline = True

data_path = "../data/"
feature_path = "../Task-4/results/"
save_path = "./results/"

os.makedirs(save_path, exist_ok=True)

# 读取离线评估数据

if offline:
    train_clicks = pd.read_csv(
        data_path + "train_click_log.csv",
        usecols=[
            "user_id",
            "click_article_id",
            "click_timestamp"
        ]
    )

    # 复用已确认的固定离线划分
    fit_clicks, query_clicks, targets = prepare_evaluation(
        train_clicks
    )

    # 检查验证用户数量
    assert targets["user_id"].nunique() == 50_000

    # 验证用户必须拥有可见点击历史
    assert targets["user_id"].isin(
        query_clicks["user_id"]
    ).all()

    print("拟合点击数据:", fit_clicks.shape)
    print("验证用户历史:", query_clicks.shape)
    print("固定验证标签:", targets.shape)


# 读取 Task 4 生成的排序特征

id_dtypes = {
    "user_id": "int64",
    "click_article_id": "int64"
}

# 排序训练特征
trn_user_item_feats_df = pd.read_csv(
    feature_path + "trn_user_item_feats_df.csv",
    dtype=id_dtypes
)

# 固定离线验证特征
if offline:
    val_user_item_feats_df = pd.read_csv(
        feature_path + "val_user_item_feats_df.csv",
        dtype=id_dtypes
    )

required_cols = [
    "user_id",
    "click_article_id",
    "score",
    "rank",
    "label"
]

assert trn_user_item_feats_df[required_cols].notna().all().all()

# 训练标签必须是0或1
assert trn_user_item_feats_df["label"].isin([0, 1]).all()

# 同一用户不能出现重复候选文章
assert not trn_user_item_feats_df.duplicated(
    ["user_id", "click_article_id"]
).any()

if offline:
    assert val_user_item_feats_df[required_cols].notna().all().all()

    assert val_user_item_feats_df["label"].isin([0, 1]).all()

    assert not val_user_item_feats_df.duplicated(
        ["user_id", "click_article_id"]
    ).any()

    # 验证候选用户必须属于固定验证集
    assert val_user_item_feats_df["user_id"].isin(
        targets["user_id"]
    ).all()

    # 训练和验证特征结构必须一致
    assert (
        trn_user_item_feats_df.columns.tolist()
        == val_user_item_feats_df.columns.tolist()
    )

# Task 4 保存的验证特征包含label，
# 但Task 5固定离线评估不能使用这些标签拟合模型。

if offline:
    val_user_item_feats_df = val_user_item_feats_df.drop(
        columns=["label"]
    )

# 排序训练标签继续保留，用于后续监督学习
# 后续构造训练输入X时，再单独排除label。

rank_train_users = int(
    trn_user_item_feats_df["user_id"].nunique()
)

print("\n========== 数据检查 ==========")

print("训练特征:", trn_user_item_feats_df.shape)
print("训练用户数:", rank_train_users)

print(
    "训练正样本数:",
    int((trn_user_item_feats_df["label"] == 1).sum())
)

print(
    "训练负样本数:",
    int((trn_user_item_feats_df["label"] == 0).sum())
)

if offline:
    print("验证特征:", val_user_item_feats_df.shape)
    print("固定验证用户数:", targets["user_id"].nunique())
    print(
        "有候选的验证用户数:",
        val_user_item_feats_df["user_id"].nunique()
    )

print("\n训练特征列:")
print(trn_user_item_feats_df.columns.tolist())

print("\n验证特征列:")
if offline:
    print(val_user_item_feats_df.columns.tolist())

print("\n数据读取完成")

拟合点击数据: (1062623, 3)
验证用户历史: (227284, 3)
固定验证标签: (50000, 3)

========== 数据检查 ==========
训练特征: (301170, 28)
训练用户数: 179657
训练正样本数: 99775
训练负样本数: 201395
验证特征: (2500000, 27)
固定验证用户数: 50000
有候选的验证用户数: 50000

训练特征列:
['user_id', 'click_article_id', 'sim0', 'time_diff0', 'word_diff0', 'sim_max', 'sim_min', 'sim_sum', 'sim_mean', 'score', 'rank', 'label', 'click_size', 'time_diff_mean', 'active_level', 'click_environment', 'click_deviceGroup', 'click_os', 'click_country', 'click_region', 'click_referrer_type', 'user_time_hob1', 'user_time_hob2', 'words_hbo', 'category_id', 'created_at_ts', 'words_count', 'is_cat_hab']

验证特征列:
['user_id', 'click_article_id', 'sim0', 'time_diff0', 'word_diff0', 'sim_max', 'sim_min', 'sim_sum', 'sim_mean', 'score', 'rank', 'click_size', 'time_diff_mean', 'active_level', 'click_environment', 'click_deviceGroup', 'click_os', 'click_country', 'click_region', 'click_referrer_type', 'user_time_hob1', 'user_time_hob2', 'words_hbo', 'category_id', 'created_at_ts', 'words

In [3]:
# ============================================================
# 返回排序后的结果
# ============================================================

# 1. 排序分数归一化
#  Min-Max 归一化
def norm_sim(sim_df, weight=0.0):
    min_sim = sim_df.min()
    max_sim = sim_df.max()

    if max_sim == min_sim:
        sim_df = sim_df.apply(lambda sim: 1.0)
    else:
        sim_df = sim_df.apply(
            lambda sim: 1.0 * (sim - min_sim)
            / (max_sim - min_sim)
        )

    sim_df = sim_df.apply(lambda sim: sim + weight)
    return sim_df


# 2. 返回排序后的结果
def submit(recall_df, topk=5, model_name='model'):
    """
    根据模型预测分数生成推荐结果。

    输入:
        user_id, click_article_id, pred_score

    输出:
        Top50长表：用于固定离线评估
        TopK宽表：用于查看推荐结果或线上提交
    """

    recall_df = recall_df[
        ['user_id', 'click_article_id', 'pred_score']
    ].copy()

    # 检查重复候选
    assert not recall_df.duplicated(
        ['user_id', 'click_article_id']
    ).any()

    # 检查预测分数
    assert np.isfinite(recall_df['pred_score']).all()

    # 按用户和预测分数排序
    # 同分时按文章ID升序排列，与公共评估规则一致
    recall_df = recall_df.sort_values(
        by=['user_id', 'pred_score', 'click_article_id'],
        ascending=[True, False, True],
        kind='mergesort'
    ).reset_index(drop=True)

    # 计算每个用户内部的预测排名
    recall_df['recall_rank'] = (
        recall_df.groupby('user_id').cumcount() + 1
    )

    # ========================================================
    # 3. 保存Top50长表，用于离线评估
    # ========================================================

    top50_df = recall_df[
        recall_df['recall_rank'] <= 50
    ][
        ['user_id', 'click_article_id',
         'pred_score', 'recall_rank']
    ].copy()

    save_predictions(
        top50_df,
        save_path + model_name + '_top50.csv'
    )

    # ========================================================
    # 4. 转换成TopK宽表
    # ========================================================

    topk_df = recall_df[
        recall_df['recall_rank'] <= topk
    ][
        ['user_id', 'click_article_id', 'recall_rank']
    ]

    submit_df = (
        topk_df
        .pivot(
            index='user_id',
            columns='recall_rank',
            values='click_article_id'
        )
        .reindex(columns=range(1, topk + 1))
        .rename(columns=lambda k: f'article_{k}')
        .reset_index()
    )

    # 使用可空整数，避免缺失候选导致文章ID变为float
    article_cols = [
        f'article_{i}' for i in range(1, topk + 1)
    ]

    submit_df[article_cols] = (
        submit_df[article_cols].astype('Int64')
    )

    # 保存TopK宽表
    save_predictions(
        submit_df,
        save_path + model_name + f'_top{topk}.csv'
    )

    return top50_df, submit_df

## LGB排序模型

In [5]:
# ============================================================
# LGB排序模型
# ============================================================

# 1. 复制训练及验证特征
trn_user_item_feats_df_rank_model = trn_user_item_feats_df.copy()

if offline:
    val_user_item_feats_df_rank_model = val_user_item_feats_df.copy()


# ============================================================
# 2. 定义模型特征列
# ============================================================

# 保留原正文的23个特征
lgb_cols = [
    'sim0', 'time_diff0', 'word_diff0',
    'sim_max', 'sim_min', 'sim_sum', 'sim_mean',
    'score', 'click_size', 'time_diff_mean',
    'active_level',
    'click_environment', 'click_deviceGroup',
    'click_os', 'click_country', 'click_region',
    'click_referrer_type',
    'user_time_hob1', 'user_time_hob2',
    'words_hbo',
    'category_id', 'created_at_ts', 'words_count'
]

# 检查特征是否齐全
missing_cols = [
    col for col in lgb_cols
    if col not in trn_user_item_feats_df_rank_model.columns
    or col not in val_user_item_feats_df_rank_model.columns
]

if missing_cols:
    raise ValueError(f"缺少排序特征: {missing_cols}")


# ============================================================
# 3. 按用户排序，构造训练分组
# ============================================================

trn_user_item_feats_df_rank_model.sort_values(
    by=['user_id'],
    kind='mergesort',
    inplace=True
)

trn_user_item_feats_df_rank_model.reset_index(
    drop=True,
    inplace=True
)

g_train = (
    trn_user_item_feats_df_rank_model
    .groupby('user_id', sort=False)
    .size()
    .values
)

# 确保组大小与训练样本行数一致
assert g_train.sum() == len(trn_user_item_feats_df_rank_model)

# 训练数据至少要包含正样本
assert (trn_user_item_feats_df_rank_model['label'] == 1).any()

rank_train_users = int(len(g_train))


# ============================================================
# 4. 定义LGB排序模型
# ============================================================

# 保留原正文超参数，先不进行调参
lgb_ranker = lgb.LGBMRanker(
    boosting_type='gbdt',
    num_leaves=31,
    reg_alpha=0.0,
    reg_lambda=1,
    max_depth=-1,
    n_estimators=100,
    subsample=0.7,
    colsample_bytree=0.7,
    subsample_freq=1,
    learning_rate=0.01,
    min_child_weight=50,
    random_state=2018,
    n_jobs=16
)


# ============================================================
# 5. 模型训练
# ============================================================

# 不使用固定外层验证标签进行Early Stopping
lgb_ranker.fit(
    trn_user_item_feats_df_rank_model[lgb_cols],
    trn_user_item_feats_df_rank_model['label'],
    group=g_train
)

print("LGB排序模型训练完成")
print("训练样本数:", len(trn_user_item_feats_df_rank_model))
print("训练用户数:", rank_train_users)
print("训练轮数:", lgb_ranker.n_estimators_)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.013576 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4201
[LightGBM] [Info] Number of data points in the train set: 301170, number of used features: 23
LGB排序模型训练完成
训练样本数: 301170
训练用户数: 179657
训练轮数: 100


In [6]:
# ============================================================
#  LGB排序模型预测
# ============================================================

# 对固定外层验证用户的候选文章进行打分
val_user_item_feats_df_rank_model['pred_score'] = (
    lgb_ranker.predict(
        val_user_item_feats_df_rank_model[lgb_cols]
    )
)

# 保留模型原始分数，为后续模型融合准备
rank_results = val_user_item_feats_df_rank_model[
    ['user_id', 'click_article_id', 'pred_score']
].copy()

save_predictions(
    rank_results,
    save_path + 'lgb_ranker_score.csv'
)


# ============================================================
# 7. 根据模型分数重新排序
# ============================================================

# 复用上一主题已经定义的submit()
lgb_top50, lgb_top5 = submit(
    rank_results,
    topk=5,
    model_name='lgb_ranker'
)


# ============================================================
# 8. 使用公共离线评估模块
# ============================================================

metrics = evaluate_csv(
    save_path + 'lgb_ranker_top50.csv',
    '../cache/offline_eval/targets.csv',
    ks=[5, 50],
    train_users=rank_train_users,
    output_path=save_path + 'lgb_ranker_metrics.json'
)

print("\n========== LGBRanker离线评估 ==========")

for name, value in metrics.items():
    print(f"{name}: {value}")


========== LGBRanker离线评估 ==========
train_users: 179657
validation_users: 50000
R@5: 0.3727
R@50: 0.61496
MRR@5: 0.2220826666666667
MRR@50: 0.2465682653890897


## LGB排序模型五折交叉验证

In [7]:
# ============================================================
# LGBRanker 五折交叉验证
# ============================================================

# 1. 按用户划分五折（保留原正文实现）
def get_kfold_users(trn_df, n=5):
    user_ids = trn_df['user_id'].unique()
    user_set = [user_ids[i::n] for i in range(n)]
    return user_set


# 2. 训练及外层验证特征
k_fold = 5

trn_df = trn_user_item_feats_df_rank_model
val_df = val_user_item_feats_df_rank_model

user_set = get_kfold_users(trn_df, n=k_fold)

# 保存各折OOF预测
score_list = []

# 五个模型对固定外层验证候选的预测分数累加
sub_preds = np.zeros(val_df.shape[0], dtype=np.float64)

# 记录各折训练用户数量
fold_train_users = []

# ============================================================
# 3. 五折交叉验证
# ============================================================

for n_fold, valid_user in enumerate(user_set):

    print(f"\n========== Fold {n_fold + 1}/{k_fold} ==========")

    # 按用户划分训练折和OOF验证折
    train_idx = trn_df[
        ~trn_df['user_id'].isin(valid_user)
    ].copy()

    valid_idx = trn_df[
        trn_df['user_id'].isin(valid_user)
    ].copy()

    # 排序模型要求相同用户的候选连续排列
    train_idx.sort_values(
        by=['user_id'],
        kind='mergesort',
        inplace=True
    )

    valid_idx.sort_values(
        by=['user_id'],
        kind='mergesort',
        inplace=True
    )

    # 训练数据的用户分组
    g_train = (
        train_idx.groupby('user_id', sort=False)
        .size()
        .values
    )

    assert g_train.sum() == len(train_idx)
    assert train_idx['label'].nunique() == 2

    fold_train_users.append(len(g_train))

    # ========================================================
    # 4. 定义模型（与上一节超参数一致）
    # ========================================================

    fold_ranker = lgb.LGBMRanker(
        boosting_type='gbdt',
        num_leaves=31,
        reg_alpha=0.0,
        reg_lambda=1,
        max_depth=-1,
        n_estimators=100,
        subsample=0.7,
        colsample_bytree=0.7,
        subsample_freq=1,
        learning_rate=0.01,
        min_child_weight=50,
        random_state=2018,
        n_jobs=16
    )

    # ========================================================
    # 5. 模型训练
    # ========================================================

    fold_ranker.fit(
        train_idx[lgb_cols],
        train_idx['label'],
        group=g_train
    )

    # ========================================================
    # 6. 预测当前OOF验证折
    # ========================================================

    valid_idx['pred_score'] = fold_ranker.predict(
        valid_idx[lgb_cols]
    )

    score_list.append(
        valid_idx[
            ['user_id', 'click_article_id', 'pred_score']
        ].copy()
    )

    # ========================================================
    # 7. 预测固定外层验证集
    # ========================================================

    # 注意：五个模型都会预测同一份外层验证候选
    sub_preds += fold_ranker.predict(
        val_df[lgb_cols]
    )

    print("训练用户数:", len(g_train))
    print("训练候选数:", len(train_idx))
    print("OOF验证用户数:", len(valid_user))
    print("OOF验证候选数:", len(valid_idx))

    # 释放当前折中间变量
    del train_idx, valid_idx, fold_ranker
    gc.collect()

print("\n五折训练完成")


========== Fold 1/5 ==========
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002255 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4195
[LightGBM] [Info] Number of data points in the train set: 240825, number of used features: 23
训练用户数: 143725
训练候选数: 240825
OOF验证用户数: 35932
OOF验证候选数: 60345

========== Fold 2/5 ==========
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008278 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4199
[LightGBM] [Info] Number of data points in the train set: 240884, number of used features: 23
训练用户数: 143725
训练候选数: 240884
OOF验证用户数: 35932
OOF验证候选数: 60286

========== Fold 3/5 ==========
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008295 seconds.
You can set `force_col_wise=true` to

In [8]:
# ============================================================
# 保存LGBRanker的OOF特征
# ============================================================

# 1. 合并五折OOF预测
score_df_ = pd.concat(
    score_list,
    axis=0,
    ignore_index=True
)

# 每个训练候选应当恰好预测一次
assert len(score_df_) == len(trn_df)

assert not score_df_.duplicated(
    ['user_id', 'click_article_id']
).any()

# 2. 将OOF分数与训练标签合并
score_df = trn_df[
    ['user_id', 'click_article_id', 'label']
].merge(
    score_df_,
    how='left',
    on=['user_id', 'click_article_id'],
    validate='one_to_one'
)

assert score_df['pred_score'].notna().all()

# 3. 根据OOF分数生成用户内部排名
score_df = score_df.sort_values(
    ['user_id', 'pred_score', 'click_article_id'],
    ascending=[True, False, True],
    kind='mergesort'
).reset_index(drop=True)

score_df['pred_rank'] = (
    score_df.groupby('user_id').cumcount() + 1
)

# 4. 保存OOF特征，供后续Stacking使用
save_predictions(
    score_df[
        ['user_id', 'click_article_id',
         'pred_score', 'pred_rank', 'label']
    ],
    save_path + 'trn_lgb_ranker_feats.csv'
)

print("OOF特征已保存")
print("OOF样本数:", len(score_df))
print("OOF用户数:", score_df['user_id'].nunique())

OOF特征已保存
OOF样本数: 301170
OOF用户数: 179657


## LGB排序模型五折评估

In [9]:
# ============================================================
# 五折平均预测及固定离线评估
# ============================================================

# 1. 五个模型的预测分数求平均
val_ranker_feats_df = val_df[
    ['user_id', 'click_article_id']
].copy()

val_ranker_feats_df['pred_score'] = sub_preds / k_fold

# 2. 计算外层验证候选的排序名次
val_ranker_feats_df = val_ranker_feats_df.sort_values(
    ['user_id', 'pred_score', 'click_article_id'],
    ascending=[True, False, True],
    kind='mergesort'
).reset_index(drop=True)

val_ranker_feats_df['pred_rank'] = (
    val_ranker_feats_df.groupby('user_id').cumcount() + 1
)

# 3. 保存五折平均预测特征
# 与原文tst_lgb_ranker_feats.csv对应
# 当前使用固定离线验证集，因此命名为val
save_predictions(
    val_ranker_feats_df[
        ['user_id', 'click_article_id',
         'pred_score', 'pred_rank']
    ],
    save_path + 'val_lgb_ranker_feats.csv'
)

# 4. 复用submit函数生成排序结果
rank_results = val_ranker_feats_df[
    ['user_id', 'click_article_id', 'pred_score']
].copy()

lgb_cv_top50, lgb_cv_top5 = submit(
    rank_results,
    topk=5,
    model_name='lgb_ranker_cv'
)

# 5. 统计五折模型合计参与拟合的去重用户
# 每位训练用户参与四个折模型的拟合
cv_train_users = int(trn_df['user_id'].nunique())

# 6. 固定外层离线评估
cv_metrics = evaluate_csv(
    save_path + 'lgb_ranker_cv_top50.csv',
    '../cache/offline_eval/targets.csv',
    ks=[5, 50],
    train_users=cv_train_users,
    output_path=save_path + 'lgb_ranker_cv_metrics.json'
)

print("\n========== LGBRanker五折评估 ==========")

for name, value in cv_metrics.items():
    print(f"{name}: {value}")


========== LGBRanker五折评估 ==========
train_users: 179657
validation_users: 50000
R@5: 0.37218
R@50: 0.61496
MRR@5: 0.22273700000000002
MRR@50: 0.2472920158031532


## LGB分类模型

In [10]:
# ============================================================
# LGB分类模型
# ============================================================

# ============================================================
# 1. 准备训练和验证特征
# ============================================================

# 复用上一节排序模型的数据和特征列
trn_user_item_feats_df_cls_model = (
    trn_user_item_feats_df_rank_model.copy()
)

val_user_item_feats_df_cls_model = (
    val_user_item_feats_df_rank_model.copy()
)

# 当前首先完成固定离线评估
assert offline is True

# 训练集必须同时包含正负样本
assert trn_user_item_feats_df_cls_model['label'].nunique() == 2

# 外层验证标签不能参与训练
assert 'label' not in val_user_item_feats_df_cls_model.columns

# 检查模型输入特征
assert set(lgb_cols).issubset(
    trn_user_item_feats_df_cls_model.columns
)

assert set(lgb_cols).issubset(
    val_user_item_feats_df_cls_model.columns
)

# 统计实际参与模型拟合的用户数
cls_train_users = int(
    trn_user_item_feats_df_cls_model['user_id'].nunique()
)


# ============================================================
# 2. 定义LGB分类模型
# ============================================================

# 保留原正文的主要模型超参数
lgb_classifier = lgb.LGBMClassifier(
    boosting_type='gbdt',
    num_leaves=31,
    reg_alpha=0.0,
    reg_lambda=1,
    max_depth=-1,
    n_estimators=500,
    subsample=0.7,
    colsample_bytree=0.7,
    subsample_freq=1,
    learning_rate=0.01,
    min_child_weight=50,
    random_state=2018,
    n_jobs=16
)


# ============================================================
# 3. 模型训练
# ============================================================

# 使用Task 4已经生成的排序训练标签
# 不使用固定外层验证集进行Early Stopping
lgb_classifier.fit(
    trn_user_item_feats_df_cls_model[lgb_cols],
    trn_user_item_feats_df_cls_model['label']
)

print("LGB分类模型训练完成")
print("训练样本数:", len(trn_user_item_feats_df_cls_model))
print("训练用户数:", cls_train_users)
print("训练轮数:", lgb_classifier.n_estimators_)


# ============================================================
# 4. 预测固定外层验证集
# ============================================================

# predict_proba返回每个类别的预测概率
# [:, 1]表示正类label=1的预测概率
val_user_item_feats_df_cls_model['pred_score'] = (
    lgb_classifier.predict_proba(
        val_user_item_feats_df_cls_model[lgb_cols]
    )[:, 1]
)

# 保留用户ID、文章ID和模型分数
rank_results = val_user_item_feats_df_cls_model[
    ['user_id', 'click_article_id', 'pred_score']
].copy()

# 保存原始预测结果，供后续模型融合使用
save_predictions(
    rank_results,
    save_path + 'lgb_cls_score.csv'
)


# ============================================================
# 5. 生成排序结果
# ============================================================

# 复用此前已经实现的submit()
lgb_cls_top50, lgb_cls_top5 = submit(
    rank_results,
    topk=5,
    model_name='lgb_cls'
)


# ============================================================
# 6. 固定离线评估
# ============================================================

cls_metrics = evaluate_csv(
    save_path + 'lgb_cls_top50.csv',
    '../cache/offline_eval/targets.csv',
    ks=[5, 50],
    train_users=cls_train_users,
    output_path=save_path + 'lgb_cls_metrics.json'
)

print("\n========== LGBClassifier离线评估 ==========")

for name, value in cls_metrics.items():
    print(f"{name}: {value}")

[LightGBM] [Info] Number of positive: 99775, number of negative: 201395
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.010615 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4201
[LightGBM] [Info] Number of data points in the train set: 301170, number of used features: 23
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.331291 -> initscore=-0.702351
[LightGBM] [Info] Start training from score -0.702351
LGB分类模型训练完成
训练样本数: 301170
训练用户数: 179657
训练轮数: 500

========== LGBClassifier离线评估 ==========
train_users: 179657
validation_users: 50000
R@5: 0.37834
R@50: 0.61496
MRR@5: 0.22576666666666667
MRR@50: 0.2499147961574813


In [ ]:
# ============================================================
# Task 5：LGB分类模型五折交叉验证
# ============================================================

# 1. 准备训练及固定外层验证数据
cls_k_fold = 5

cls_trn_df = trn_user_item_feats_df_cls_model
cls_val_df = val_user_item_feats_df_cls_model

assert offline is True
assert 'label' not in cls_val_df.columns

# 复用上一主题定义的用户五折划分函数
cls_user_set = get_kfold_users(
    cls_trn_df, n=cls_k_fold
)

# 保存每折的OOF预测
score_list_cls = []

# 五折模型对固定外层验证候选的预测分数累加
sub_preds_cls = np.zeros(
    cls_val_df.shape[0],
    dtype=np.float64
)

# 保持外层验证特征顺序不变
cls_val_x = cls_val_df[lgb_cols]

# ============================================================
# 2. 五折交叉验证
# ============================================================

for n_fold, valid_user in enumerate(cls_user_set):

    print(f"\n========== Fold {n_fold + 1}/{cls_k_fold} ==========")

    # 按用户划分训练折和OOF验证折
    train_idx = cls_trn_df[
        ~cls_trn_df['user_id'].isin(valid_user)
    ].copy()

    valid_idx = cls_trn_df[
        cls_trn_df['user_id'].isin(valid_user)
    ].copy()

    # 检查训练折的标签
    assert train_idx['label'].nunique() == 2

    # ========================================================
    # 3. 定义分类模型
    # ========================================================

    # 保留原正文五折版本的100轮配置
    fold_classifier = lgb.LGBMClassifier(
        boosting_type='gbdt',
        num_leaves=31,
        reg_alpha=0.0,
        reg_lambda=1,
        max_depth=-1,
        n_estimators=100,
        subsample=0.7,
        colsample_bytree=0.7,
        subsample_freq=1,
        learning_rate=0.01,
        min_child_weight=50,
        random_state=2018,
        n_jobs=16
    )

    # ========================================================
    # 4. 训练模型
    # ========================================================

    # 分类模型不需要group
    # 不使用当前验证折标签进行Early Stopping
    fold_classifier.fit(
        train_idx[lgb_cols],
        train_idx['label']
    )

    # ========================================================
    # 5. 当前折的OOF预测
    # ========================================================

    valid_idx['pred_score'] = (
        fold_classifier.predict_proba(
            valid_idx[lgb_cols]
        )[:, 1]
    )

    # 按用户内部预测分数降序排序
    # 同分时按文章ID升序排序
    valid_idx = valid_idx.sort_values(
        ['user_id', 'pred_score', 'click_article_id'],
        ascending=[True, False, True],
        kind='mergesort'
    ).reset_index(drop=True)

    # 生成当前用户内的预测排名
    valid_idx['pred_rank'] = (
        valid_idx.groupby('user_id').cumcount() + 1
    )

    # 保存当前折OOF特征
    score_list_cls.append(
        valid_idx[
            ['user_id', 'click_article_id',
             'pred_score', 'pred_rank']
        ].copy()
    )

    # ========================================================
    # 6. 预测固定外层验证集
    # ========================================================

    # 无论offline是否为True，都需要为外层验证生成预测
    sub_preds_cls += fold_classifier.predict_proba(
        cls_val_x
    )[:, 1]

    print("训练用户数:", train_idx['user_id'].nunique())
    print("训练样本数:", len(train_idx))
    print("OOF用户数:", valid_idx['user_id'].nunique())
    print("OOF样本数:", len(valid_idx))

    del train_idx, valid_idx, fold_classifier
    gc.collect()

print("\nLGBClassifier五折训练完成")

## LGB分类模型五折交叉验证

In [11]:
# ============================================================
# LGB分类模型五折交叉验证
# ============================================================

# 1. 准备训练及固定外层验证数据
cls_k_fold = 5

cls_trn_df = trn_user_item_feats_df_cls_model
cls_val_df = val_user_item_feats_df_cls_model

assert offline is True
assert 'label' not in cls_val_df.columns

# 复用上一主题定义的用户五折划分函数
cls_user_set = get_kfold_users(
    cls_trn_df, n=cls_k_fold
)

# 保存每折的OOF预测
score_list_cls = []

# 五折模型对固定外层验证候选的预测分数累加
sub_preds_cls = np.zeros(
    cls_val_df.shape[0],
    dtype=np.float64
)

# 保持外层验证特征顺序不变
cls_val_x = cls_val_df[lgb_cols]

# ============================================================
# 2. 五折交叉验证
# ============================================================

for n_fold, valid_user in enumerate(cls_user_set):

    print(f"\n========== Fold {n_fold + 1}/{cls_k_fold} ==========")

    # 按用户划分训练折和OOF验证折
    train_idx = cls_trn_df[
        ~cls_trn_df['user_id'].isin(valid_user)
    ].copy()

    valid_idx = cls_trn_df[
        cls_trn_df['user_id'].isin(valid_user)
    ].copy()

    # 检查训练折的标签
    assert train_idx['label'].nunique() == 2

    # ========================================================
    # 3. 定义分类模型
    # ========================================================

    # 保留原正文五折版本的100轮配置
    fold_classifier = lgb.LGBMClassifier(
        boosting_type='gbdt',
        num_leaves=31,
        reg_alpha=0.0,
        reg_lambda=1,
        max_depth=-1,
        n_estimators=100,
        subsample=0.7,
        colsample_bytree=0.7,
        subsample_freq=1,
        learning_rate=0.01,
        min_child_weight=50,
        random_state=2018,
        n_jobs=16
    )

    # ========================================================
    # 4. 训练模型
    # ========================================================

    # 分类模型不需要group
    # 不使用当前验证折标签进行Early Stopping
    fold_classifier.fit(
        train_idx[lgb_cols],
        train_idx['label']
    )

    # ========================================================
    # 5. 当前折的OOF预测
    # ========================================================

    valid_idx['pred_score'] = (
        fold_classifier.predict_proba(
            valid_idx[lgb_cols]
        )[:, 1]
    )

    # 按用户内部预测分数降序排序
    # 同分时按文章ID升序排序
    valid_idx = valid_idx.sort_values(
        ['user_id', 'pred_score', 'click_article_id'],
        ascending=[True, False, True],
        kind='mergesort'
    ).reset_index(drop=True)

    # 生成当前用户内的预测排名
    valid_idx['pred_rank'] = (
        valid_idx.groupby('user_id').cumcount() + 1
    )

    # 保存当前折OOF特征
    score_list_cls.append(
        valid_idx[
            ['user_id', 'click_article_id',
             'pred_score', 'pred_rank']
        ].copy()
    )

    # ========================================================
    # 6. 预测固定外层验证集
    # ========================================================

    # 无论offline是否为True，都需要为外层验证生成预测
    sub_preds_cls += fold_classifier.predict_proba(
        cls_val_x
    )[:, 1]

    print("训练用户数:", train_idx['user_id'].nunique())
    print("训练样本数:", len(train_idx))
    print("OOF用户数:", valid_idx['user_id'].nunique())
    print("OOF样本数:", len(valid_idx))

    del train_idx, valid_idx, fold_classifier
    gc.collect()

print("\nLGBClassifier五折训练完成")


========== Fold 1/5 ==========
[LightGBM] [Info] Number of positive: 79727, number of negative: 161098
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008980 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4195
[LightGBM] [Info] Number of data points in the train set: 240825, number of used features: 23
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.331058 -> initscore=-0.703405
[LightGBM] [Info] Start training from score -0.703405
训练用户数: 143725
训练样本数: 240825
OOF用户数: 35932
OOF样本数: 60345

========== Fold 2/5 ==========
[LightGBM] [Info] Number of positive: 79893, number of negative: 160991
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.009771 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4199
[LightGBM] [Info] Number of data points in the train set: 240884, number of used features: 23
[LightGBM] [Info] [bin

In [12]:
# ============================================================
# 保存LGBClassifier的OOF特征
# ============================================================

# 1. 合并五折OOF结果
score_df_cls_ = pd.concat(
    score_list_cls,
    axis=0,
    ignore_index=True
)

# 2. 检查OOF覆盖情况
assert len(score_df_cls_) == len(cls_trn_df)

assert not score_df_cls_.duplicated(
    ['user_id', 'click_article_id']
).any()

# 3. 合并训练标签
score_df_cls = cls_trn_df[
    ['user_id', 'click_article_id', 'label']
].merge(
    score_df_cls_,
    on=['user_id', 'click_article_id'],
    how='left',
    validate='one_to_one'
)

assert score_df_cls['pred_score'].notna().all()
assert score_df_cls['pred_rank'].notna().all()

# 4. 保存OOF特征
save_predictions(
    score_df_cls[
        ['user_id', 'click_article_id',
         'pred_score', 'pred_rank', 'label']
    ],
    save_path + 'trn_lgb_cls_feats.csv'
)

print("LGBClassifier OOF特征保存完成")
print("训练样本数:", len(score_df_cls))
print("训练用户数:", score_df_cls['user_id'].nunique())

LGBClassifier OOF特征保存完成
训练样本数: 301170
训练用户数: 179657


## LGB分类模型五折评估

In [13]:
# ============================================================
# LGBClassifier五折平均预测与离线评估
# ============================================================

# ============================================================
# 1. 计算五折平均预测分数
# ============================================================

val_cls_feats_df = cls_val_df[
    ['user_id', 'click_article_id']
].copy()

val_cls_feats_df['pred_score'] = (
    sub_preds_cls / cls_k_fold
)

# ============================================================
# 2. 计算验证候选的用户内部排名
# ============================================================

val_cls_feats_df = val_cls_feats_df.sort_values(
    ['user_id', 'pred_score', 'click_article_id'],
    ascending=[True, False, True],
    kind='mergesort'
).reset_index(drop=True)

val_cls_feats_df['pred_rank'] = (
    val_cls_feats_df.groupby('user_id').cumcount() + 1
)

# ============================================================
# 3. 保存固定外层验证特征
# ============================================================

save_predictions(
    val_cls_feats_df[
        ['user_id', 'click_article_id',
         'pred_score', 'pred_rank']
    ],
    save_path + 'val_lgb_cls_feats.csv'
)

# ============================================================
# 4. 生成Top50和Top5结果
# ============================================================

rank_results_cls = val_cls_feats_df[
    ['user_id', 'click_article_id', 'pred_score']
].copy()

lgb_cls_cv_top50, lgb_cls_cv_top5 = submit(
    rank_results_cls,
    topk=5,
    model_name='lgb_cls_cv'
)

# ============================================================
# 5. 固定离线评估
# ============================================================

# 五个折模型合计实际参与拟合的去重用户
cls_cv_train_users = int(
    cls_trn_df['user_id'].nunique()
)

cls_cv_metrics = evaluate_csv(
    save_path + 'lgb_cls_cv_top50.csv',
    '../cache/offline_eval/targets.csv',
    ks=[5, 50],
    train_users=cls_cv_train_users,
    output_path=save_path + 'lgb_cls_cv_metrics.json'
)

print("\n========== LGBClassifier五折评估 ==========")

for name, value in cls_cv_metrics.items():
    print(f"{name}: {value}")


========== LGBClassifier五折评估 ==========
train_users: 179657
validation_users: 50000
R@5: 0.3657
R@50: 0.61496
MRR@5: 0.21248899999999998
MRR@50: 0.23728457430648087


## DIN模型

In [14]:
# ============================================================
# DIN模型
# 1. 用户的历史点击行为列表
# ============================================================

from metric.data import split_last_click


# ============================================================
# 1. 复用Task 4已确认的内层数据划分
# ============================================================

# fit_clicks已经排除了固定外层验证标签
# 再留出内层末次点击，作为排序模型训练标签
rank_hist, rank_targets = split_last_click(fit_clicks)


# ============================================================
# 2. 构建用户历史点击列表
# ============================================================

def get_hist_click_df(click_df):
    """
    根据点击日志构建用户历史行为序列。

    返回:
        user_id
        hist_click_article_id
    """

    # 按用户和点击时间排序
    click_df = click_df.sort_values(
        ['user_id', 'click_timestamp'],
        kind='mergesort'
    )

    # 按用户聚合点击文章，得到历史行为列表
    hist_click = (
        click_df.groupby('user_id', sort=False)
        ['click_article_id']
        .agg(list)
        .reset_index(name='hist_click_article_id')
    )

    return hist_click


# ============================================================
# 3. 分别生成训练与验证历史行为
# ============================================================

# 排序训练历史：不包含rank_targets
his_behavior_trn_df = get_hist_click_df(
    rank_hist
)

# 外层验证历史：不包含固定targets
his_behavior_val_df = get_hist_click_df(
    query_clicks
)


# ============================================================
# 4. 准备DIN训练及验证特征
# ============================================================

trn_user_item_feats_df_din_model = (
    trn_user_item_feats_df.copy()
)

val_user_item_feats_df_din_model = (
    val_user_item_feats_df.copy()
)

# 记录合并前样本数量
trn_rows = len(trn_user_item_feats_df_din_model)
val_rows = len(val_user_item_feats_df_din_model)


# ============================================================
# 5. 合并历史点击序列
# ============================================================

# DIN训练特征
trn_user_item_feats_df_din_model = (
    trn_user_item_feats_df_din_model.merge(
        his_behavior_trn_df,
        on='user_id',
        how='left',
        validate='many_to_one'
    )
)

# DIN外层验证特征
val_user_item_feats_df_din_model = (
    val_user_item_feats_df_din_model.merge(
        his_behavior_val_df,
        on='user_id',
        how='left',
        validate='many_to_one'
    )
)


# ============================================================
# 6. 检查历史序列与样本对应关系
# ============================================================

# 合并不能改变候选样本数量
assert len(trn_user_item_feats_df_din_model) == trn_rows
assert len(val_user_item_feats_df_din_model) == val_rows

# 每条候选必须存在对应的历史行为列表
assert trn_user_item_feats_df_din_model[
    'hist_click_article_id'
].map(
    lambda x: isinstance(x, list) and len(x) > 0
).all()

assert val_user_item_feats_df_din_model[
    'hist_click_article_id'
].map(
    lambda x: isinstance(x, list) and len(x) > 0
).all()

# 训练标签保留，验证标签不能进入模型
assert 'label' in trn_user_item_feats_df_din_model.columns
assert 'label' not in val_user_item_feats_df_din_model.columns

# 检查内外层被留出的点击事件没有进入对应历史
click_cols = [
    'user_id',
    'click_article_id',
    'click_timestamp'
]

assert rank_targets.merge(
    rank_hist,
    on=click_cols
).empty

assert targets.merge(
    query_clicks,
    on=click_cols
).empty


# ============================================================
# 7. 统计历史行为序列信息
# ============================================================

trn_hist_len = his_behavior_trn_df[
    'hist_click_article_id'
].map(len)

val_hist_len = his_behavior_val_df[
    'hist_click_article_id'
].map(len)

print("DIN训练特征:", trn_user_item_feats_df_din_model.shape)
print("DIN验证特征:", val_user_item_feats_df_din_model.shape)

print("训练历史用户数:", len(his_behavior_trn_df))
print("验证历史用户数:", len(his_behavior_val_df))

print("\n训练历史序列长度统计:")
print(trn_hist_len.describe())

print("\n验证历史序列长度统计:")
print(val_hist_len.describe())

print("\nDIN历史行为构建完成")

DIN训练特征: (301170, 29)
DIN验证特征: (2500000, 28)
训练历史用户数: 200000
验证历史用户数: 50000

训练历史序列长度统计:
count    200000.000000
mean          4.414830
std           6.488553
min           1.000000
25%           1.000000
50%           2.000000
75%           5.000000
max         240.000000
Name: hist_click_article_id, dtype: float64

验证历史序列长度统计:
count    50000.000000
mean         4.545680
std          6.497676
min          1.000000
25%          1.000000
50%          3.000000
75%          5.000000
max        179.000000
Name: hist_click_article_id, dtype: float64

DIN历史行为构建完成


## DIN 模型简介

下面尝试使用DIN模型， DIN的全称是Deep Interest Network， 这是阿里2018年基于前面的深度学习模型无法表达用户多样化的兴趣而提出的一个模型， 它可以通过考虑【给定的候选广告】和【用户的历史行为】的相关性，来计算用户兴趣的表示向量。具体来说就是通过引入局部激活单元，通过软搜索历史行为的相关部分来关注相关的用户兴趣，并采用加权和来获得有关候选广告的用户兴趣的表示。与候选广告相关性较高的行为会获得较高的激活权重，并支配着用户兴趣。该表示向量在不同广告上有所不同，大大提高了模型的表达能力。所以该模型对于此次新闻推荐的任务也比较适合， 我们在这里通过当前的候选文章与用户历史点击文章的相关性来计算用户对于文章的兴趣。

我们这里直接调包来使用这个模型， 关于这个模型的详细细节部分我们会在下一期的推荐系统组队学习中给出。下面说一下该模型如何具体使用：deepctr的函数原型如下：

> def DIN(dnn_feature_columns, history_feature_list, dnn_use_bn=False, dnn_hidden_units=(200, 80), dnn_activation='relu', att_hidden_size=(80, 40), att_activation="dice", att_weight_normalization=False, l2_reg_dnn=0, l2_reg_embedding=1e-6, dnn_dropout=0, seed=1024, task='binary'):
> - dnn_feature_columns: 特征列， 包含数据所有特征的列表
> - history_feature_list: 用户历史行为列， 反应用户历史行为的特征的列表
> - dnn_use_bn: 是否使用BatchNormalization
> - dnn_hidden_units: 全连接层网络的层数和每一层神经元的个数， 一个列表或者元组
> - dnn_activation_relu: 全连接网络的激活单元类型
> - att_hidden_size: 注意力层的全连接网络的层数和每一层神经元的个数
> - att_activation: 注意力层的激活单元类型
> - att_weight_normalization: 是否归一化注意力得分
> - l2_reg_dnn: 全连接网络的正则化系数
> - l2_reg_embedding: embedding向量的正则化稀疏
> - dnn_dropout: 全连接网络的神经元的失活概率
> - task: 任务， 可以是分类， 也可是是回归


在具体使用的时候， 我们必须要传入特征列和历史行为列， 但是再传入之前， 我们需要进行一下特征列的预处理。具体如下：

1. 首先，我们要处理数据集， 得到数据， 由于我们是基于用户过去的行为去预测用户是否点击当前文章， 所以我们需要把数据的特征列划分成数值型特征， 离散型特征和历史行为特征列三部分， 对于每一部分， DIN模型的处理会有不同

   - 对于离散型特征， 在我们的数据集中就是那些类别型的特征， 比如user_id这种， 这种类别型特征， 我们首先要经过embedding处理得到每个特征的低维稠密型表示， 既然要经过embedding， 那么我们就需要为每一列的类别特征的取值建立一个字典，并指明embedding维度， 所以在使用deepctr的DIN模型准备数据的时候， 我们需要通过SparseFeat函数指明这些类别型特征, 这个函数的传入参数就是列名， 列的唯一取值(建立字典用)和embedding维度。

   - 对于用户历史行为特征列， 比如文章id， 文章的类别等这种， 同样的我们需要先经过embedding处理， 只不过和上面不一样的地方是，对于这种特征， 我们在得到每个特征的embedding表示之后， 还需要通过一个Attention_layer计算用户的历史行为和当前候选文章的相关性以此得到当前用户的embedding向量， 这个向量就可以基于当前的候选文章与用户过去点击过得历史文章的相似性的程度来反应用户的兴趣， 并且随着用户的不同的历史点击来变化，去动态的模拟用户兴趣的变化过程。这类特征对于每个用户都是一个历史行为序列， 对于每个用户， 历史行为序列长度会不一样， 可能有的用户点击的历史文章多，有的点击的历史文章少， 所以我们还需要把这个长度统一起来， 在为DIN模型准备数据的时候， 我们首先要通过SparseFeat函数指明这些类别型特征， 然后还需要通过VarLenSparseFeat函数再进行序列填充， 使得每个用户的历史序列一样长， 所以这个函数参数中会有个maxlen，来指明序列的最大长度是多少。

   - 对于连续型特征列， 我们只需要用DenseFeat函数来指明列名和维度即可。

2. 处理完特征列之后， 我们把相应的数据与列进行对应，就得到了最后的数据

In [17]:
# ============================================================
# DIN特征输入构建
# ============================================================

from deepctr.feature_column import (
    SparseFeat,
    DenseFeat,
    VarLenSparseFeat,
    get_feature_names
)

import tensorflow as tf

from tf_keras.preprocessing.sequence import pad_sequences


# ============================================================
# 1. 复用上一主题的DIN训练和验证数据
# ============================================================

assert offline is True

trn_din_df = trn_user_item_feats_df_din_model
val_din_df = val_user_item_feats_df_din_model

assert 'label' in trn_din_df.columns
assert 'label' not in val_din_df.columns


# ============================================================
# 2. 特征划分（保留原正文）
# ============================================================

sparse_fea = [
    'user_id',
    'click_article_id',
    'category_id',
    'click_environment',
    'click_deviceGroup',
    'click_os',
    'click_country',
    'click_region',
    'click_referrer_type',
    'is_cat_hab'
]

behavior_fea = ['click_article_id']

hist_behavior_fea = ['hist_click_article_id']

dense_fea = [
    'sim0', 'time_diff0', 'word_diff0',
    'sim_max', 'sim_min', 'sim_sum', 'sim_mean',
    'score', 'rank', 'click_size', 'time_diff_mean',
    'active_level', 'user_time_hob1', 'user_time_hob2',
    'words_hbo', 'words_count'
]

# 沿用正文实际调用时的设置
DIN_EMB_DIM = 32
DIN_MAX_LEN = 50

# 检查输入特征是否齐全
required_feats = sparse_fea + dense_fea + hist_behavior_fea

for df in (trn_din_df, val_din_df):
    missing = set(required_feats) - set(df.columns)
    if missing:
        raise ValueError(f"DIN缺少输入特征: {sorted(missing)}")

warnings.filterwarnings('ignore')

In [18]:
# ============================================================
# 3. 离散特征编码
# ============================================================

sparse_maps = {}
vocab_sizes = {}

# 文章ID使用完整文章目录建立统一词表
article_ids = pd.read_csv(
    "../data/articles.csv",
    usecols=['article_id'],
    dtype={'article_id': 'int64'}
)['article_id'].drop_duplicates()

article2idx = {
    int(article): idx + 2
    for idx, article in enumerate(article_ids)
}

sparse_maps['click_article_id'] = article2idx
vocab_sizes['click_article_id'] = len(article2idx) + 2


# 其他离散特征只根据训练集建立词表
for feat in sparse_fea:

    if feat == 'click_article_id':
        continue

    categories = trn_din_df[feat].dropna().unique()

    sparse_maps[feat] = {
        value: idx + 2
        for idx, value in enumerate(categories)
    }

    vocab_sizes[feat] = len(sparse_maps[feat]) + 2

print("文章词表大小:", vocab_sizes['click_article_id'])
print("用户词表大小:", vocab_sizes['user_id'])

文章词表大小: 364049
用户词表大小: 179659


In [19]:
# ============================================================
# 4. 连续特征归一化
# ============================================================

# 只根据训练数据计算缺失值填补规则
train_dense = (
    trn_din_df[dense_fea]
    .replace([np.inf, -np.inf], np.nan)
    .astype('float32')
)

dense_fill_values = train_dense.median().fillna(0.0)

# 只在训练数据上拟合归一化参数
dense_scaler = MinMaxScaler()

dense_scaler.fit(
    train_dense.fillna(dense_fill_values)
)

del train_dense
gc.collect()

20

In [20]:
# ============================================================
# 5. 定义DeepCTR特征列
# ============================================================

# 离散特征
sparse_feature_columns = [
    SparseFeat(
        feat,
        vocabulary_size=vocab_sizes[feat],
        embedding_dim=DIN_EMB_DIM
    )
    for feat in sparse_fea
]

# 连续特征
dense_feature_columns = [
    DenseFeat(feat, 1)
    for feat in dense_fea
]

# 用户历史行为序列
var_feature_columns = [
    VarLenSparseFeat(
        SparseFeat(
            feat,
            vocabulary_size=vocab_sizes['click_article_id'],
            embedding_dim=DIN_EMB_DIM,
            embedding_name='click_article_id'
        ),
        maxlen=DIN_MAX_LEN
    )
    for feat in hist_behavior_fea
]

# 拼接所有DIN输入特征列
dnn_feature_columns = (
    sparse_feature_columns
    + dense_feature_columns
    + var_feature_columns
)

# 获取模型输入字段名称
feature_names = get_feature_names(dnn_feature_columns)

In [24]:
# ============================================================
# 6. 准备DIN模型输入
# ============================================================

def get_din_feats_columns(df):
    """
    根据统一的特征列、编码表与归一化规则，
    构建DeepCTR的DIN模型输入字典。

    返回:
        x: 模型输入字典
        dnn_feature_columns: 特征列定义
    """

    # 连续特征使用训练阶段拟合的归一化规则
    dense_df = (
        df[dense_fea]
        .replace([np.inf, -np.inf], np.nan)
        .astype('float32')
        .fillna(dense_fill_values)
    )

    dense_matrix = dense_scaler.transform(
        dense_df
    ).astype('float32')

    x = {}

    for name in get_feature_names(dnn_feature_columns):

        # 1. 历史行为序列特征
        if name in hist_behavior_fea:

            # 每位用户只编码一次历史序列
            user_histories = (
                df[['user_id', name]]
                .drop_duplicates('user_id')
            )

            hist_by_user = {
                user: [
                    article2idx.get(int(article), 1)
                    for article in hist
                ]
                for user, hist in user_histories.itertuples(
                    index=False, name=None
                )
            }

            # 恢复到每条用户-候选文章记录
            his_list = df['user_id'].map(
                hist_by_user
            ).tolist()

            # 统一历史序列长度
            x[name] = pad_sequences(
                his_list,
                maxlen=DIN_MAX_LEN,
                padding='post',
                truncating='pre',
                value=0,
                dtype='int32'
            )

        # 2. 离散特征
        elif name in sparse_fea:

            x[name] = (
                df[name]
                .map(sparse_maps[name])
                .fillna(1)
                .astype('int32')
                .to_numpy()
                .reshape(-1, 1)
            )

        # 3. 连续特征
        elif name in dense_fea:

            col_idx = dense_fea.index(name)

            x[name] = dense_matrix[
                :, col_idx
            ].reshape(-1, 1)

    return x, dnn_feature_columns


# ============================================================
# 7. 构建训练、验证输入
# ============================================================

x_trn, dnn_feature_columns = get_din_feats_columns(
    trn_din_df
)

y_trn = trn_din_df['label'].to_numpy(
    dtype='float32'
).reshape(-1, 1)

# 固定外层验证输入
x_val, _ = get_din_feats_columns(
    val_din_df
)

# 不提取y_val，不使用固定外层标签训练


# ============================================================
# 8. 输入完整性检查
# ============================================================

assert set(x_trn) == set(feature_names)
assert set(x_val) == set(feature_names)

assert x_trn['hist_click_article_id'].shape == (
    len(trn_din_df), DIN_MAX_LEN
)

assert x_val['hist_click_article_id'].shape == (
    len(val_din_df), DIN_MAX_LEN
)

assert len(y_trn) == len(trn_din_df)

print("DIN训练样本数:", len(y_trn))
print("DIN验证候选数:", len(val_din_df))

print(
    "训练历史序列输入形状:",
    x_trn['hist_click_article_id'].shape
)

print(
    "验证历史序列输入形状:",
    x_val['hist_click_article_id'].shape
)

print("DIN输入构建完成")

DIN训练样本数: 301170
DIN验证候选数: 2500000
训练历史序列输入形状: (301170, 50)
验证历史序列输入形状: (2500000, 50)
DIN输入构建完成


In [22]:
# ============================================================
# 9. 保存DIN预处理规则,方便后续五折验证和模型融合
# ============================================================

din_preprocess = {
    'sparse_maps': sparse_maps,
    'vocab_sizes': vocab_sizes,
    'dense_fill_values': dense_fill_values,
    'dense_scaler': dense_scaler,
    'sparse_fea': sparse_fea,
    'dense_fea': dense_fea,
    'behavior_fea': behavior_fea,
    'hist_behavior_fea': hist_behavior_fea,
    'emb_dim': DIN_EMB_DIM,
    'max_len': DIN_MAX_LEN
}

with open(
    save_path + 'din_preprocess.pkl', 'wb'
) as f:
    pickle.dump(din_preprocess, f)

print("DIN预处理规则已保存")

DIN预处理规则已保存


## DIN模型建立、训练与离线评估

In [25]:
# ============================================================
# DIN单模型训练与固定离线评估
# ============================================================

import tensorflow as tf
from deepctr.models import DIN

# ============================================================
# 1. 训练配置
# ============================================================

assert offline is True

DIN_EPOCHS = 2
DIN_BATCH_SIZE = 256

# 训练集保留标签，外层验证集不参与训练
assert 'label' in trn_din_df.columns
assert 'label' not in val_din_df.columns

# 输入数据必须与候选样本一一对应
assert len(y_trn) == len(trn_din_df)
assert len(x_val['click_article_id']) == len(val_din_df)

# 实际参与模型训练的去重用户数
din_train_users = int(
    trn_din_df['user_id'].nunique()
)


# ============================================================
# 2. 建立DIN模型
# ============================================================

# 复用原正文的DIN网络结构
model = DIN(
    dnn_feature_columns,
    history_feature_list=behavior_fea
)

# 查看模型结构
model.summary()


# ============================================================
# 3. 编译模型
# ============================================================

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[
        'binary_crossentropy',
        tf.keras.metrics.AUC(name='auc')
    ]
)


# ============================================================
# 4. 训练模型
# ============================================================

# 不传入固定外层验证标签
# 暂时固定训练2个epoch，不进行Early Stopping
history = model.fit(
    x_trn,
    y_trn,
    batch_size=DIN_BATCH_SIZE,
    epochs=DIN_EPOCHS,
    verbose=1,
    shuffle=True
)

print("\nDIN模型训练完成")
print("训练候选数:", len(y_trn))
print("训练用户数:", din_train_users)


# ============================================================
# 5. 预测固定外层验证候选
# ============================================================

din_pred = model.predict(
    x_val,
    batch_size=DIN_BATCH_SIZE,
    verbose=1
).reshape(-1)

assert len(din_pred) == len(val_din_df)
assert np.isfinite(din_pred).all()

# 将预测分数与原始用户、文章ID对应
din_rank_results = val_din_df[
    ['user_id', 'click_article_id']
].copy()

din_rank_results['pred_score'] = din_pred

# 保存原始预测分数
save_predictions(
    din_rank_results,
    save_path + 'din_rank_score.csv'
)


# ============================================================
# 6. 生成Top50和Top5推荐结果
# ============================================================

# 复用第二个主题实现的submit函数
din_top50, din_top5 = submit(
    din_rank_results,
    topk=5,
    model_name='din'
)


# ============================================================
# 7. 使用公共模块进行固定离线评估
# ============================================================

din_metrics = evaluate_csv(
    save_path + 'din_top50.csv',
    '../cache/offline_eval/targets.csv',
    ks=[5, 50],
    train_users=din_train_users,
    output_path=save_path + 'din_metrics.json'
)

print("\n========== DIN离线评估 ==========")

for name, value in din_metrics.items():
    print(f"{name}: {value}")

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ user_id             │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ click_article_id    │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ category_id         │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ click_environment   │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ click_deviceGroup   │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ click_os            │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ click_country       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ click_region        │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ click_referrer_type │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ is_cat_hab          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ hist_click_article… │ (None, 50)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sparse_emb_user_id  │ (None, 1, 32)     │  5,749,088 │ user_id[0][0]     │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sparse_seq_emb_his… │ (None, 1, 32)     │ 11,649,568 │ click_article_id… │
│ (Embedding)         │                   │            │ hist_click_artic… │
│                     │                   │            │ click_article_id… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sparse_emb_categor… │ (None, 1, 32)     │      8,032 │ category_id[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sparse_emb_click_e… │ (None, 1, 32)     │        160 │ click_environmen… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sparse_emb_click_d… │ (None, 1, 32)     │        192 │ click_deviceGrou… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 17,544,833 (66.93 MB)

 Trainable params: 17,544,833 (66.93 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/2
1177/1177 ━━━━━━━━━━━━━━━━━━━━ 121s 99ms/step - auc: 0.8707 - binary_crossentropy: 0.4188 - loss: 0.4200
Epoch 2/2
1177/1177 ━━━━━━━━━━━━━━━━━━━━ 127s 108ms/step - auc: 0.9598 - binary_crossentropy: 0.2399 - loss: 0.2444

DIN模型训练完成
训练候选数: 301170
训练用户数: 179657
9766/9766 ━━━━━━━━━━━━━━━━━━━━ 82s 8ms/step

========== DIN离线评估 ==========
train_users: 179657
validation_users: 50000
R@5: 0.30834
R@50: 0.61496
MRR@5: 0.17031933333333335
MRR@50: 0.196505130224955


## DIN五折验证预处理

In [26]:
# ============================================================
# DIN五折——折内特征预处理
# ============================================================

def fit_din_fold_preprocess(train_fold):
    """
    仅使用当前训练折拟合DIN预处理规则。
    文章ID统一使用已建立的article2idx。
    """

    fold_sparse_maps = {
        'click_article_id': article2idx
    }

    fold_vocab_sizes = {
        'click_article_id': len(article2idx) + 2
    }

    # 1. 其他离散特征：只在当前训练折建立编码表
    for feat in sparse_fea:

        if feat == 'click_article_id':
            continue

        values = train_fold[feat].dropna().unique()

        fold_sparse_maps[feat] = {
            value: idx + 2
            for idx, value in enumerate(values)
        }

        fold_vocab_sizes[feat] = (
            len(fold_sparse_maps[feat]) + 2
        )

    # 2. 连续特征：仅使用当前训练折拟合
    dense_train = (
        train_fold[dense_fea]
        .replace([np.inf, -np.inf], np.nan)
        .astype('float32')
    )

    fold_fill_values = (
        dense_train.median().fillna(0.0)
    )

    fold_scaler = MinMaxScaler()

    fold_scaler.fit(
        dense_train.fillna(fold_fill_values)
    )

    # 3. 构造与前一主题一致的DeepCTR特征列
    sparse_columns = [
        SparseFeat(
            feat,
            vocabulary_size=fold_vocab_sizes[feat],
            embedding_dim=DIN_EMB_DIM
        )
        for feat in sparse_fea
    ]

    dense_columns = [
        DenseFeat(feat, 1)
        for feat in dense_fea
    ]

    var_columns = [
        VarLenSparseFeat(
            SparseFeat(
                feat,
                vocabulary_size=fold_vocab_sizes[
                    'click_article_id'
                ],
                embedding_dim=DIN_EMB_DIM,
                embedding_name='click_article_id'
            ),
            maxlen=DIN_MAX_LEN
        )
        for feat in hist_behavior_fea
    ]

    fold_feature_columns = (
        sparse_columns
        + dense_columns
        + var_columns
    )

    return {
        'sparse_maps': fold_sparse_maps,
        'vocab_sizes': fold_vocab_sizes,
        'fill_values': fold_fill_values,
        'scaler': fold_scaler,
        'feature_columns': fold_feature_columns
    }

In [27]:
def transform_din_fold(df, preprocess):
    """
    将DataFrame转换为DIN输入字典。
    不重新拟合任何编码或归一化参数。
    """

    x = {}

    fold_sparse_maps = preprocess['sparse_maps']

    # ========================================================
    # 1. 离散特征编码
    # ========================================================

    for feat in sparse_fea:

        x[feat] = (
            df[feat]
            .map(fold_sparse_maps[feat])
            .fillna(1)
            .to_numpy(dtype=np.int32)
            .reshape(-1, 1)
        )

    # ========================================================
    # 2. 连续特征转换
    # ========================================================

    dense_data = (
        df[dense_fea]
        .replace([np.inf, -np.inf], np.nan)
        .astype('float32')
        .fillna(preprocess['fill_values'])
    )

    dense_matrix = preprocess['scaler'].transform(
        dense_data
    ).astype('float32')

    for idx, feat in enumerate(dense_fea):
        x[feat] = dense_matrix[
            :, idx
        ].reshape(-1, 1)

    # ========================================================
    # 3. 历史点击序列编码
    # ========================================================

    for feat in hist_behavior_fea:

        # 当前数据中每位用户只处理一次历史序列
        user_histories = (
            df[['user_id', feat]]
            .drop_duplicates('user_id')
        )

        hist_by_user = {
            user: [
                article2idx.get(int(article), 1)
                for article in hist
            ]
            for user, hist in user_histories.itertuples(
                index=False,
                name=None
            )
        }

        hist_list = (
            df['user_id']
            .map(hist_by_user)
            .tolist()
        )

        x[feat] = pad_sequences(
            hist_list,
            maxlen=DIN_MAX_LEN,
            padding='post',
            truncating='pre',
            value=0,
            dtype='int32'
        )

    # ========================================================
    # 4. 输入检查
    # ========================================================

    feature_names_fold = get_feature_names(
        preprocess['feature_columns']
    )

    assert set(x.keys()) == set(feature_names_fold)

    return x

## DIN五折交叉验证

In [28]:
# ============================================================
# DIN五折交叉验证
# ============================================================

from deepctr.models import DIN
import tensorflow as tf

# ============================================================
# 1. 准备数据与运行参数
# ============================================================

assert offline is True

din_cv_trn_df = trn_din_df
din_cv_val_df = val_din_df

assert 'label' in din_cv_trn_df.columns
assert 'label' not in din_cv_val_df.columns

din_k_fold = 5
din_cv_epochs = 2
din_cv_batch_size = 256

# 外层验证预测每次最多处理10万条候选
DIN_PRED_CHUNK = 100_000

# 复用前面已经定义的按用户划分函数
din_user_set = get_kfold_users(
    din_cv_trn_df,
    n=din_k_fold
)

# 保存每折的OOF预测
score_list_din = []

# 保存五个模型对外层验证数据的预测分数之和
sub_preds_din = np.zeros(
    len(din_cv_val_df),
    dtype=np.float64
)

# 记录各折实际训练用户数量
din_fold_train_users = []


# ============================================================
# 2. 五折交叉验证
# ============================================================

for n_fold, valid_user in enumerate(din_user_set):

    print(
        f"\n========== DIN Fold {n_fold + 1}/{din_k_fold} =========="
    )

    # --------------------------------------------------------
    # 2.1 按用户划分训练折和OOF折
    # --------------------------------------------------------

    valid_mask = (
        din_cv_trn_df['user_id']
        .isin(valid_user)
        .to_numpy()
    )

    train_idx = din_cv_trn_df.iloc[
        np.flatnonzero(~valid_mask)
    ].copy()

    valid_idx = din_cv_trn_df.iloc[
        np.flatnonzero(valid_mask)
    ].copy()

    assert train_idx['label'].nunique() == 2

    assert set(train_idx['user_id']).isdisjoint(
        set(valid_idx['user_id'])
    )

    # --------------------------------------------------------
    # 2.2 当前折独立拟合预处理规则
    # --------------------------------------------------------

    fold_preprocess = fit_din_fold_preprocess(
        train_idx
    )

    # 当前训练折输入
    x_fold_trn = transform_din_fold(
        train_idx,
        fold_preprocess
    )

    # 当前OOF折输入
    x_fold_val = transform_din_fold(
        valid_idx,
        fold_preprocess
    )

    # 标签必须为(N,1)，与DIN输出一致
    y_fold_trn = (
        train_idx['label']
        .to_numpy(dtype='float32')
        .reshape(-1, 1)
    )

    # --------------------------------------------------------
    # 2.3 每折重新建立独立DIN模型
    # --------------------------------------------------------

    tf.keras.backend.clear_session()

    fold_model = DIN(
        fold_preprocess['feature_columns'],
        history_feature_list=behavior_fea,
        seed=1024 + n_fold
    )

    fold_model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=[
            tf.keras.metrics.AUC(name='auc')
        ]
    )

    # --------------------------------------------------------
    # 2.4 训练当前折模型
    # --------------------------------------------------------

    # 不使用OOF验证折标签训练或选择Epoch
    fold_model.fit(
        x_fold_trn,
        y_fold_trn,
        batch_size=din_cv_batch_size,
        epochs=din_cv_epochs,
        verbose=1,
        shuffle=True
    )

    # --------------------------------------------------------
    # 2.5 生成当前OOF折预测
    # --------------------------------------------------------

    valid_idx['pred_score'] = (
        fold_model.predict(
            x_fold_val,
            batch_size=din_cv_batch_size,
            verbose=0
        ).reshape(-1)
    )

    # 计算用户内部排名
    valid_idx = valid_idx.sort_values(
        ['user_id', 'pred_score', 'click_article_id'],
        ascending=[True, False, True],
        kind='mergesort'
    ).reset_index(drop=True)

    valid_idx['pred_rank'] = (
        valid_idx.groupby('user_id').cumcount() + 1
    )

    score_list_din.append(
        valid_idx[
            ['user_id', 'click_article_id',
             'pred_score', 'pred_rank']
        ].copy()
    )

    # --------------------------------------------------------
    # 2.6 预测固定外层验证集
    # --------------------------------------------------------

    # 使用当前折的预处理规则转换外层验证特征
    # 分块生成输入，避免一次性存储250万条历史序列

    for start in range(
        0,
        len(din_cv_val_df),
        DIN_PRED_CHUNK
    ):

        end = min(
            start + DIN_PRED_CHUNK,
            len(din_cv_val_df)
        )

        val_chunk = din_cv_val_df.iloc[start:end]

        x_chunk = transform_din_fold(
            val_chunk,
            fold_preprocess
        )

        chunk_pred = fold_model.predict(
            x_chunk,
            batch_size=din_cv_batch_size,
            verbose=0
        ).reshape(-1)

        sub_preds_din[start:end] += chunk_pred

        del x_chunk, val_chunk, chunk_pred

    # --------------------------------------------------------
    # 2.7 记录当前折信息
    # --------------------------------------------------------

    fold_train_users = int(
        train_idx['user_id'].nunique()
    )

    din_fold_train_users.append(fold_train_users)

    print("训练用户数:", fold_train_users)
    print("训练样本数:", len(train_idx))
    print("OOF用户数:", valid_idx['user_id'].nunique())
    print("OOF样本数:", len(valid_idx))

    # --------------------------------------------------------
    # 2.8 释放当前折占用的内存
    # --------------------------------------------------------

    del (
        train_idx,
        valid_idx,
        x_fold_trn,
        x_fold_val,
        y_fold_trn,
        fold_model,
        fold_preprocess
    )

    tf.keras.backend.clear_session()
    gc.collect()

print("\nDIN五折训练完成")


========== DIN Fold 1/5 ==========
Epoch 1/2
941/941 ━━━━━━━━━━━━━━━━━━━━ 101s 103ms/step - auc: 0.8621 - loss: 0.4316
Epoch 2/2
941/941 ━━━━━━━━━━━━━━━━━━━━ 100s 106ms/step - auc: 0.9386 - loss: 0.2988
训练用户数: 143725
训练样本数: 240825
OOF用户数: 35932
OOF样本数: 60345

========== DIN Fold 2/5 ==========
Epoch 1/2
941/941 ━━━━━━━━━━━━━━━━━━━━ 94s 96ms/step - auc: 0.8632 - loss: 0.4303
Epoch 2/2
941/941 ━━━━━━━━━━━━━━━━━━━━ 91s 97ms/step - auc: 0.9533 - loss: 0.2613
训练用户数: 143725
训练样本数: 240884
OOF用户数: 35932
OOF样本数: 60286

========== DIN Fold 3/5 ==========
Epoch 1/2
942/942 ━━━━━━━━━━━━━━━━━━━━ 96s 98ms/step - auc: 0.8606 - loss: 0.4336
Epoch 2/2
942/942 ━━━━━━━━━━━━━━━━━━━━ 95s 101ms/step - auc: 0.9455 - loss: 0.2822
训练用户数: 143726
训练样本数: 240974
OOF用户数: 35931
OOF样本数: 60196

========== DIN Fold 4/5 ==========
Epoch 1/2
942/942 ━━━━━━━━━━━━━━━━━━━━ 100s 102ms/step - auc: 0.8648 - loss: 0.4278
Epoch 2/2
942/942 ━━━━━━━━━━━━━━━━━━━━ 97s 103ms/step - auc: 0.9557 - loss: 0.2554
训练用户数: 143726
训练样本数: 240

In [29]:
# ============================================================
# 保存DIN训练侧OOF特征
# ============================================================

# 1. 合并五折OOF结果
score_df_din_ = pd.concat(
    score_list_din,
    axis=0,
    ignore_index=True
)

# 2. 验证每条训练候选恰好预测一次
assert len(score_df_din_) == len(din_cv_trn_df)

assert not score_df_din_.duplicated(
    ['user_id', 'click_article_id']
).any()

# 3. 与训练标签合并
score_df_din = din_cv_trn_df[
    ['user_id', 'click_article_id', 'label']
].merge(
    score_df_din_,
    on=['user_id', 'click_article_id'],
    how='left',
    validate='one_to_one'
)

assert len(score_df_din) == len(din_cv_trn_df)
assert score_df_din['pred_score'].notna().all()
assert score_df_din['pred_rank'].notna().all()

# 4. 保存OOF特征
save_predictions(
    score_df_din[
        ['user_id', 'click_article_id',
         'pred_score', 'pred_rank', 'label']
    ],
    save_path + 'trn_din_cls_feats.csv'
)

print("DIN OOF特征保存完成")
print("OOF样本数:", len(score_df_din))
print("OOF用户数:", score_df_din['user_id'].nunique())

DIN OOF特征保存完成
OOF样本数: 301170
OOF用户数: 179657


## DIN五折预测与离线评估

In [30]:
# ============================================================
# DIN五折平均预测与固定离线评估
# ============================================================

# ============================================================
# 1. 五折模型平均预测
# ============================================================

val_din_cv_feats_df = din_cv_val_df[
    ['user_id', 'click_article_id']
].copy()

val_din_cv_feats_df['pred_score'] = (
    sub_preds_din / din_k_fold
)

assert np.isfinite(
    val_din_cv_feats_df['pred_score']
).all()


# ============================================================
# 2. 计算外层验证用户内部排名
# ============================================================

val_din_cv_feats_df = val_din_cv_feats_df.sort_values(
    ['user_id', 'pred_score', 'click_article_id'],
    ascending=[True, False, True],
    kind='mergesort'
).reset_index(drop=True)

val_din_cv_feats_df['pred_rank'] = (
    val_din_cv_feats_df.groupby('user_id').cumcount() + 1
)


# ============================================================
# 3. 保存外层验证预测特征
# ============================================================

save_predictions(
    val_din_cv_feats_df[
        ['user_id', 'click_article_id',
         'pred_score', 'pred_rank']
    ],
    save_path + 'val_din_cls_feats.csv'
)


# ============================================================
# 4. 生成Top50和Top5
# ============================================================

din_cv_rank_results = val_din_cv_feats_df[
    ['user_id', 'click_article_id', 'pred_score']
].copy()

din_cv_top50, din_cv_top5 = submit(
    din_cv_rank_results,
    topk=5,
    model_name='din_cv'
)


# ============================================================
# 5. 固定离线评估
# ============================================================

# 五折模型合计参与拟合的去重用户数
din_cv_train_users = int(
    din_cv_trn_df['user_id'].nunique()
)

din_cv_metrics = evaluate_csv(
    save_path + 'din_cv_top50.csv',
    '../cache/offline_eval/targets.csv',
    ks=[5, 50],
    train_users=din_cv_train_users,
    output_path=save_path + 'din_cv_metrics.json'
)

print("\n========== DIN五折离线评估 ==========")

for name, value in din_cv_metrics.items():
    print(f"{name}: {value}")


========== DIN五折离线评估 ==========
train_users: 179657
validation_users: 50000
R@5: 0.31902
R@50: 0.61496
MRR@5: 0.17454
MRR@50: 0.20129050541236151


## 模型融合

### 加权融合

In [31]:
# ============================================================
# 模型融合——加权融合
# ============================================================

assert offline is True

# ============================================================
# 1. 选择基础模型的预测结果
# ============================================================

# False：使用原正文的三个单模型预测
# True ：使用三个模型的五折平均预测
use_cv = False

if use_cv:
    score_files = {
        'lgb_ranker': 'val_lgb_ranker_feats.csv',
        'lgb_cls': 'val_lgb_cls_feats.csv',
        'din_ranker': 'val_din_cls_feats.csv'
    }
    model_name = 'ensemble_fuse_cv'

else:
    score_files = {
        'lgb_ranker': 'lgb_ranker_score.csv',
        'lgb_cls': 'lgb_cls_score.csv',
        'din_ranker': 'din_rank_score.csv'
    }
    model_name = 'ensemble_fuse'


# ============================================================
# 2. 读取模型分数
# ============================================================

id_cols = ['user_id', 'click_article_id']

def load_model_score(filename):

    df = pd.read_csv(
        save_path + filename,
        usecols=id_cols + ['pred_score'],
        dtype={
            'user_id': 'int64',
            'click_article_id': 'int64',
            'pred_score': 'float64'
        }
    )

    if df.empty:
        raise ValueError(f"{filename} 没有预测数据")

    if df.duplicated(id_cols).any():
        raise ValueError(f"{filename} 存在重复候选文章")

    if df.isna().any().any():
        raise ValueError(f"{filename} 存在缺失值")

    if not np.isfinite(df['pred_score']).all():
        raise ValueError(f"{filename} 存在无效预测分数")

    return df


# 保留原文rank_model字典结构
rank_model = {
    name: load_model_score(filename)
    for name, filename in score_files.items()
}

print("基础模型预测结果：")

for name, df in rank_model.items():
    print(name, df.shape)

基础模型预测结果：
lgb_ranker (2500000, 3)
lgb_cls (2500000, 3)
din_ranker (2500000, 3)


In [32]:
# ============================================================
# 3. 加权融合函数
# ============================================================

def get_ensumble_predict_topk(
    rank_model,
    topk=5,
    model_name='ensemble_fuse',
    weights=None
):
    """
    将三个基础模型的预测分数进行加权融合。

    LGBRanker：先进行Min-Max归一化
    LGBClassifier、DIN：保留原始正类概率

    三个模型的候选文章集合必须完全一致。
    """

    # 1. 默认使用原文的等权方案
    if weights is None:
        weights = {
            'lgb_ranker': 1.0,
            'lgb_cls': 1.0,
            'din_ranker': 1.0
        }

    model_names = [
        'lgb_ranker',
        'lgb_cls',
        'din_ranker'
    ]

    if set(weights) != set(model_names):
        raise ValueError("融合权重必须对应三个基础模型")

    weight_values = np.array(
        [weights[name] for name in model_names],
        dtype=float
    )

    if (
        not np.isfinite(weight_values).all()
        or (weight_values < 0).any()
        or weight_values.sum() == 0
    ):
        raise ValueError("融合权重必须非负、有限且不能全为0")

    # ========================================================
    # 2. 处理LGBRanker分数
    # ========================================================

    lgb_ranker_df = rank_model['lgb_ranker'].copy()

    # 沿用正文归一化函数
    lgb_ranker_df['pred_score'] = norm_sim(
        lgb_ranker_df['pred_score']
    )

    # 区分不同模型的分数列
    final_recall = lgb_ranker_df.rename(
        columns={'pred_score': 'score_ranker'}
    )

    # ========================================================
    # 3. 按用户ID、文章ID合并其他模型分数
    # ========================================================

    score_names = {
        'lgb_cls': 'score_cls',
        'din_ranker': 'score_din'
    }

    for name, score_col in score_names.items():

        model_df = rank_model[name].rename(
            columns={'pred_score': score_col}
        )

        # 三个模型应当预测相同的候选集合
        if len(model_df) != len(final_recall):
            raise ValueError(
                f"{name} 的候选数量与LGBRanker不一致"
            )

        final_recall = final_recall.merge(
            model_df,
            on=id_cols,
            how='left',
            validate='one_to_one',
            sort=False
        )

        if final_recall[score_col].isna().any():
            raise ValueError(
                f"{name} 的候选集合与LGBRanker不一致"
            )

    # ========================================================
    # 4. 分数加权融合
    # ========================================================

    final_recall['pred_score'] = (
        weights['lgb_ranker'] * final_recall['score_ranker']
        + weights['lgb_cls'] * final_recall['score_cls']
        + weights['din_ranker'] * final_recall['score_din']
    )

    # 只保留后续排序所需的三列
    final_recall = final_recall[
        ['user_id', 'click_article_id', 'pred_score']
    ]

    # ========================================================
    # 5. 保存融合分数
    # ========================================================

    save_predictions(
        final_recall,
        save_path + model_name + '_score.csv'
    )

    # ========================================================
    # 6. 生成Top50和Top5
    # ========================================================

    top50_df, top5_df = submit(
        final_recall,
        topk=topk,
        model_name=model_name
    )

    return top50_df, top5_df

In [33]:
# ============================================================
# 4. 执行加权融合
# ============================================================

# 与原正文一致，三个模型等权
ensemble_weights = {
    'lgb_ranker': 1.0,
    'lgb_cls': 1.0,
    'din_ranker': 1.0
}

ensemble_top50, ensemble_top5 = get_ensumble_predict_topk(
    rank_model,
    topk=5,
    model_name=model_name,
    weights=ensemble_weights
)


# ============================================================
# 5. 固定离线评估
# ============================================================

# 融合本身不重新训练模型。
# 三个基础模型使用同一份Task 4排序训练用户，
# 因此统计基础模型实际使用过的去重训练用户。
ensemble_train_users = int(
    trn_user_item_feats_df['user_id'].nunique()
)

ensemble_metrics = evaluate_csv(
    save_path + model_name + '_top50.csv',
    '../cache/offline_eval/targets.csv',
    ks=[5, 50],
    train_users=ensemble_train_users,
    output_path=save_path + model_name + '_metrics.json'
)

print("\n========== 加权融合离线评估 ==========")

for name, value in ensemble_metrics.items():
    print(f"{name}: {value}")


========== 加权融合离线评估 ==========
train_users: 179657
validation_users: 50000
R@5: 0.36946
R@50: 0.61496
MRR@5: 0.20711966666666667
MRR@50: 0.23239922693376633


### Stacking

In [34]:
# ============================================================
# Stacking模型融合
# 1. 读取并合并基础模型的OOF预测特征
# ============================================================

from sklearn.linear_model import LogisticRegression

assert offline is True

# ============================================================
# 1. 定义基础模型文件
# ============================================================

stacking_model_files = [
    (
        'lgb_ranker',
        'trn_lgb_ranker_feats.csv',
        'val_lgb_ranker_feats.csv'
    ),
    (
        'lgb_cls',
        'trn_lgb_cls_feats.csv',
        'val_lgb_cls_feats.csv'
    ),
    (
        'din',
        'trn_din_cls_feats.csv',
        'val_din_cls_feats.csv'
    )
]

key_cols = ['user_id', 'click_article_id']

# 保留原正文的6个输入特征
feat_cols = [
    'pred_score_0', 'pred_rank_0',
    'pred_score_1', 'pred_rank_1',
    'pred_score_2', 'pred_rank_2'
]


# ============================================================
# 2. 读取并合并模型预测特征
# ============================================================

def build_stacking_features(stage='trn'):
    """
    合并三个基础模型的预测特征。

    stage='trn'：读取OOF预测，包含label
    stage='val'：读取固定外层验证预测，不包含label

    必须根据user_id、click_article_id精确关联。
    """

    if stage not in ('trn', 'val'):
        raise ValueError("stage必须是'trn'或'val'")

    final_df = None

    for idx, (model, trn_file, val_file) in enumerate(
        stacking_model_files
    ):

        filename = trn_file if stage == 'trn' else val_file

        usecols = key_cols + ['pred_score', 'pred_rank']

        if stage == 'trn':
            usecols += ['label']

        dtype = {
            'user_id': 'int64',
            'click_article_id': 'int64',
            'pred_score': 'float64',
            'pred_rank': 'float64',
            'label': 'float64'
        }

        model_df = pd.read_csv(
            save_path + filename,
            usecols=usecols,
            dtype={col: dtype[col] for col in usecols}
        )

        # 检查候选与预测分数
        if model_df.empty:
            raise ValueError(f"{filename}为空")

        if model_df.duplicated(key_cols).any():
            raise ValueError(f"{filename}存在重复候选")

        if not np.isfinite(
            model_df[['pred_score', 'pred_rank']].to_numpy()
        ).all():
            raise ValueError(f"{filename}存在无效预测值")

        if (model_df['pred_rank'] < 1).any():
            raise ValueError(f"{filename}包含无效排名")

        if stage == 'trn':
            if not model_df['label'].isin([0, 1]).all():
                raise ValueError(f"{filename}包含无效训练标签")

        # 第一个模型提供候选主表及训练标签
        if final_df is None:
            columns = key_cols + (
                ['label'] if stage == 'trn' else []
            )
            final_df = model_df[columns].copy()

        # 对模型预测特征重命名
        pred_cols = key_cols + ['pred_score', 'pred_rank']

        # 其他模型的训练标签也读取进来核对
        if stage == 'trn' and idx > 0:
            pred_cols += ['label']

        model_feats = model_df[pred_cols].rename(
            columns={
                'pred_score': f'pred_score_{idx}',
                'pred_rank': f'pred_rank_{idx}',
                'label': 'label_check'
            }
        )

        final_df = final_df.merge(
            model_feats,
            on=key_cols,
            how='left',
            validate='one_to_one',
            sort=False
        )

        # 确保三个模型预测的候选集合一致
        if (
            len(final_df) != len(model_df)
            or final_df[
                [f'pred_score_{idx}', f'pred_rank_{idx}']
            ].isna().any().any()
        ):
            raise ValueError(
                f"{filename}与其他模型的候选集合不一致"
            )

        # 三个模型的OOF训练标签必须一致
        if stage == 'trn' and idx > 0:
            if not final_df['label'].eq(
                final_df['label_check']
            ).all():
                raise ValueError(
                    f"{filename}与原始训练标签不一致"
                )

            final_df.drop(
                columns=['label_check'],
                inplace=True
            )

        print(f"{model} {stage}特征合并完成")

        del model_df, model_feats

    return final_df


# ============================================================
# 3. 合并训练OOF与固定验证预测
# ============================================================

final_trn_ranker_feats = build_stacking_features('trn')

final_val_ranker_feats = build_stacking_features('val')


# ============================================================
# 4. 检查与Task 4原始候选数量一致
# ============================================================

assert len(final_trn_ranker_feats) == len(
    trn_user_item_feats_df
)

assert len(final_val_ranker_feats) == len(
    val_user_item_feats_df
)

assert final_trn_ranker_feats[feat_cols].notna().all().all()
assert final_val_ranker_feats[feat_cols].notna().all().all()

assert 'label' in final_trn_ranker_feats.columns
assert 'label' not in final_val_ranker_feats.columns

print("\n========== Stacking输入 ==========")
print("训练特征:", final_trn_ranker_feats.shape)
print("验证特征:", final_val_ranker_feats.shape)
print("训练用户数:", final_trn_ranker_feats['user_id'].nunique())

lgb_ranker trn特征合并完成
lgb_cls trn特征合并完成
din trn特征合并完成
lgb_ranker val特征合并完成
lgb_cls val特征合并完成
din val特征合并完成

========== Stacking输入 ==========
训练特征: (301170, 9)
验证特征: (2500000, 8)
训练用户数: 179657


In [35]:
# ============================================================
# 第二层逻辑回归模型训练
# ============================================================

# ============================================================
# 1. 准备训练特征与标签
# ============================================================

trn_x = final_trn_ranker_feats[feat_cols]

trn_y = final_trn_ranker_feats['label'].astype(int)

# 固定外层验证特征
val_x = final_val_ranker_feats[feat_cols]

assert trn_y.nunique() == 2


# ============================================================
# 2. 定义第二层逻辑回归模型
# ============================================================

lr = LogisticRegression(
    max_iter=1000,
    random_state=42
)


# ============================================================
# 3. 模型训练
# ============================================================

lr.fit(
    trn_x,
    trn_y
)

# 统计实际参与第二层模型拟合的用户
stacking_train_users = int(
    final_trn_ranker_feats['user_id'].nunique()
)

print("\nStacking逻辑回归训练完成")
print("训练样本数:", len(trn_x))
print("训练用户数:", stacking_train_users)

# 查看逻辑回归学习到的系数
lr_coef = pd.Series(
    lr.coef_.ravel(),
    index=feat_cols,
    name='coefficient'
)

print("\n逻辑回归特征系数:")
print(lr_coef)
print("截距:", lr.intercept_[0])


Stacking逻辑回归训练完成
训练样本数: 301170
训练用户数: 179657

逻辑回归特征系数:
pred_score_0    1.774226
pred_rank_0    -1.491311
pred_score_1    2.359898
pred_rank_1    -0.454599
pred_score_2    3.163326
pred_rank_2     0.459132
Name: coefficient, dtype: float64
截距: -0.8056236095137974


In [36]:
# ============================================================
# 外层验证预测及评估
# ============================================================

# ============================================================
# 1. 使用第二层逻辑回归预测
# ============================================================

STACKING_PRED_CHUNK = 200_000

stacking_preds = np.empty(
    len(final_val_ranker_feats),
    dtype=np.float64
)

for start in range(
    0,
    len(final_val_ranker_feats),
    STACKING_PRED_CHUNK
):
    end = min(
        start + STACKING_PRED_CHUNK,
        len(final_val_ranker_feats)
    )

    stacking_preds[start:end] = lr.predict_proba(
        val_x.iloc[start:end]
    )[:, 1]


# ============================================================
# 2. 整理模型预测结果
# ============================================================

stacking_rank_results = final_val_ranker_feats[
    ['user_id', 'click_article_id']
].copy()

stacking_rank_results['pred_score'] = stacking_preds

assert np.isfinite(
    stacking_rank_results['pred_score']
).all()

# 保存原始Stacking预测分数
save_predictions(
    stacking_rank_results,
    save_path + 'ensemble_stacking_score.csv'
)


# ============================================================
# 3. 生成Top50、Top5预测结果
# ============================================================

stacking_top50, stacking_top5 = submit(
    stacking_rank_results,
    topk=5,
    model_name='ensemble_stacking'
)


# ============================================================
# 4. 固定离线评估
# ============================================================

stacking_metrics = evaluate_csv(
    save_path + 'ensemble_stacking_top50.csv',
    '../cache/offline_eval/targets.csv',
    ks=[5, 50],
    train_users=stacking_train_users,
    output_path=save_path + 'ensemble_stacking_metrics.json'
)

print("\n========== Stacking离线评估 ==========")

for name, value in stacking_metrics.items():
    print(f"{name}: {value}")


========== Stacking离线评估 ==========
train_users: 179657
validation_users: 50000
R@5: 0.3523
R@50: 0.61496
MRR@5: 0.20285266666666665
MRR@50: 0.2290191047519068
